In [ ]:
!pip -q install --upgrade pandas==2.2.2 tqdm
!pip install autogen-agentchat[gemini]~=0.2
import pandas as pd
print("pandas:", pd.__version__)

pandas: 2.2.2


In [ ]:
!pip install autogen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 834.0/834.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 17.5 MB/s eta 0:00:00


In [ ]:
from autogen import AssistantAgent
import time, os, json, ast
from datetime import datetime

In [ ]:
# --- Auth (one SA key that has access to BOTH projects is ideal) ---
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "..."  # <-- put your key file here

# --- Projects / locations you shared ---
# Flash 2.0 (Vertex)
GCP_PROJECT_FLASH   = "..."  # <-- put your project id here
GCP_LOCATION_FLASH  = "..."  # <-- put your location here

# Your finetuned endpoint (Vertex)
GCP_PROJECT_TUNED   = "..."  # <-- put your project id here
GCP_LOCATION_TUNED  = "..."  # <-- put your location here
TUNED_ENDPOINT_ID   = "..."  # deployed endpoint id

# ============== AutoGen llm_config ==============

# A) Native Gemini via AutoGen (api_type="google") — for Flash 2.0
config_list_gemini_flash = [{
    "model": "gemini-2.0-flash-001",
    "api_type": "google",
    "project_id": GCP_PROJECT_FLASH,
    "location": GCP_LOCATION_FLASH,
}]

# B) Finetuned endpoint via Vertex OpenAI-compatible gateway (cleanest for endpoints)
#    Uses ADC token from your SA json above.
import google.auth
import google.auth.transport.requests as gar
scopes = ["https://www.googleapis.com/auth/cloud-platform"]
creds, _ = google.auth.default(scopes=scopes)
req = gar.Request()
creds.refresh(req)

openai_style_base_tuned = (
    f"https://{GCP_LOCATION_TUNED}-aiplatform.googleapis.com/v1beta1/"
    f"projects/{GCP_PROJECT_TUNED}/locations/{GCP_LOCATION_TUNED}/endpoints/openapi"
)

config_list_gemini_tuned = [{
    "model": f"projects/{GCP_PROJECT_TUNED}/locations/{GCP_LOCATION_TUNED}/endpoints/{TUNED_ENDPOINT_ID}",
    "api_type": "google",
    "project_id": GCP_PROJECT_TUNED,
    "location": GCP_LOCATION_TUNED,
}]

# (Optional) shared generation params
LLM_TEMPERATURE = 0.7

In [ ]:
# Flash (publisher model)
llm_config_flash = {
    "config_list": config_list_gemini_flash,  # api_type: "google"
    "temperature": 0.7,                       # keep ONE knob only
    "max_output_tokens": 512,                # <- use this name
}

# Tuned (your endpoint)
llm_config_tuned = {
    "config_list": config_list_gemini_tuned,  # api_type: "google", model=<endpoint id>
    "temperature": 0.7,
    "max_output_tokens": 512,
}

In [ ]:
import google.auth, google.auth.transport.requests as gar

def get_llm_config_tuned(max_tokens=512, temperature=0.7):
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    creds.refresh(gar.Request())
    base = f"https://{GCP_LOCATION_TUNED}-aiplatform.googleapis.com/v1beta1/" \
           f"projects/{GCP_PROJECT_TUNED}/locations/{GCP_LOCATION_TUNED}/endpoints/openapi"
    return {
        "config_list": [{
            "model": f"projects/{GCP_PROJECT_TUNED}/locations/{GCP_LOCATION_TUNED}/endpoints/{TUNED_ENDPOINT_ID}",
            "api_type": "openai",
            "base_url": base,
            "api_key": creds.token,
        }],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

In [ ]:
def create_autogen_agents(
    harasser_name: str,
    harasser_role: str,
    victim_name: str,
    planning_method: str,
    conversation_for_memory: str,
    llm_config: dict,
):
    pm = (planning_method or "none").lower()
    if pm == "none":
        h_sys = f"You are an assistant. Your name is {harasser_name} and your role is {harasser_role}:"
        v_sys = f"You are an assistant. Your name is {victim_name} and your role is victim:"
    elif pm == "cot":
        h_sys = (f"You are an assistant. Your name is {harasser_name} and your role is {harasser_role}. "
                 "When responding, first think step by step about your strategy and reasoning, "
                 "then provide your response. Structure your answers with 'Thinking: ...' followed by 'Response: ...'")
        v_sys = (f"You are an assistant. Your name is {victim_name} and your role is victim. "
                 "When responding, first think step by step about your feelings and reasoning, "
                 "then provide your response. Structure your answers with 'Thinking: ...' followed by 'Response: ...'")
    elif pm == "react":
        fmt = ("Follow this format for your responses:\n"
               "Thought: Analyze the situation and think about your strategy\n"
               "Action: Decide what to say or do next\n"
               "Observation: Reflect on the potential impact of your action\n"
               "Response: Your final message to the other person")
        h_sys = f"You are an assistant. Your name is {harasser_name} and your role is {harasser_role}. {fmt}"
        v_sys = f"You are an assistant. Your name is {victim_name} and your role is victim. Respond naturally to messages you receive.{fmt}"
    elif pm == "memory":
        h_sys = f"You are an assistant. Your name is {harasser_name} and your role is {harasser_role}. Here is your past conversation {conversation_for_memory}"
        v_sys = f"You are an assistant. Your name is {victim_name} and your role is victim. Here is your past conversation {conversation_for_memory}"
    else:
        raise ValueError(f"Unknown planning_method: {planning_method}")

    h_agent = AssistantAgent(name=harasser_name, system_message=h_sys, llm_config=llm_config, is_termination_msg=lambda x: False)
    v_agent = AssistantAgent(name=victim_name,  system_message=v_sys, llm_config=llm_config, is_termination_msg=lambda x: False)
    return h_agent, v_agent

def _serialize_chat_messages(chat_messages: dict) -> str:
    out = []
    for agent, messages in chat_messages.items():
        for m in messages:
            content = (m.get("content") or "").strip()
            if not content:
                continue
            out.append({
                "agent": str(agent),
                "content": content,
                "role": m.get("role"),
                "name": m.get("name"),
            })
    return json.dumps(out, indent=4)


In [ ]:
def simulate_chat_autogen(
    harasser_name: str,
    harasser_role: str,
    victim_name: str,
    full_convo_str: str,
    initial_msg: str,
    planning_method: str,
    llm_config: dict,
    max_turns: int = 10,   # 10 rounds ≈ up to 20 total messages
) -> str:
    h_agent, v_agent = create_autogen_agents(
        harasser_name=harasser_name,
        harasser_role=harasser_role,
        victim_name=victim_name,
        planning_method=planning_method,
        conversation_for_memory=full_convo_str if planning_method.lower() == "memory" else "",
        llm_config=llm_config,
    )
    h_agent.initiate_chat(v_agent, message=initial_msg, max_turns=max_turns, stream=False, silent=True)
    return _serialize_chat_messages(h_agent.chat_messages)


In [ ]:
import contextlib, os, sys

def process_row_gemini_autogen(row, planning_method: str, use_finetuned: bool = False, max_turns: int = 10):
    try:
        # 1) Parse attributes
        attrs = json.loads(row["agent2_output_json"])
        attrs = {k.lower(): v for k, v in attrs.items()}
        harasser_name = attrs["harasser"]
        victim_name   = attrs["victim"]
        harasser_role = attrs["harassment goal"]

        # 2) Parse seed conversation & first message
        conversation = ast.literal_eval(row["agent3_output_converted"])
        initial_msg = conversation[0]["message"]
        if harasser_name in initial_msg:
            initial_msg = initial_msg.replace(harasser_name, "").strip()

        full_convo_str = "\n".join([f"{m['role']}: {m['message']}" for m in conversation])

        # 3) AutoGen constraint: names without whitespace
        harasser_name = harasser_name.split()[0]
        victim_name   = victim_name.split()[0]

        # 4) Pick config: Flash vs Finetuned
        if use_finetuned:
            llm_config = {
                "config_list": config_list_gemini_tuned,
                "temperature": 0.7,
                "max_tokens": 1024,   # give tuned room to speak
            }
        else:
            llm_config = {
                "config_list": config_list_gemini_flash,
                "temperature": 0.7,
                "max_tokens": 512,   # same for Flash if you want longer messages
            }

        # 5) Run
        return simulate_chat_autogen(
            harasser_name=harasser_name,
            harasser_role=harasser_role,
            victim_name=victim_name,
            full_convo_str=full_convo_str,
            initial_msg=initial_msg,
            planning_method=planning_method,
            llm_config=llm_config,
            max_turns=max_turns,
        )
    except Exception as e:
        return f"ERROR: {e}"


In [ ]:
from tqdm import tqdm
from datetime import datetime
import csv, os, pandas as pd
from google.colab import files   # 👈 needed for auto-download

# --- Config ---
INPUT_CSV  = "..."   # <-- set your path
OUTPUT_DIR = "..."   # <-- set your path
MAX_ROWS   = None                    # or small int for testing
STRATEGIES = ["None", "COT", "ReACT", "Memory"]

# --- Load ---
df_full = pd.read_csv(INPUT_CSV, dtype=str, engine="python")
df = df_full.head(MAX_ROWS) if (isinstance(MAX_ROWS, int) and MAX_ROWS > 0) else df_full
print(f"✅ Loaded {len(df)} rows")

RUN_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
os.makedirs(OUTPUT_DIR, exist_ok=True)
saved_files = []

RUNS = [
    ("ReACT", False, "Gemini Flash")
]

START_ROW   = 0
CHUNK_SIZE  = 100
START_PART  = 0

results = []

for strategy, use_tuned, backend in RUNS:
    print(f"\n▶ Strategy={strategy}  |  Backend={backend}")
    prefix = "tuned" if use_tuned else "flash"
    base_name = f"gemini_{prefix}_{strategy.lower()}_{RUN_TS}_{os.path.basename(INPUT_CSV)}"

    for rel_idx, (global_idx, row) in enumerate(tqdm(
        df.iloc[START_ROW:].iterrows(),
        total=len(df) - START_ROW,
        desc=f"{backend}/{strategy}"
    )):
        results.append(
            process_row_gemini_autogen(
                row,
                planning_method=strategy,
                use_finetuned=use_tuned,
                max_turns=10,
            )
        )

        # Save every CHUNK_SIZE rows
        if (global_idx + 1) % CHUNK_SIZE == 0 or (global_idx + 1) == len(df):
            df_out = df.iloc[:global_idx+1].copy()
            df_out.loc[START_ROW:global_idx, "bully_chat_history"] = results
            part_num = START_PART + (global_idx + 1 - START_ROW) // CHUNK_SIZE
            out_name = f"{base_name}.part{part_num:03d}.csv"
            out_path = os.path.join(OUTPUT_DIR, out_name)
            df_out.to_csv(out_path, index=False, quoting=csv.QUOTE_ALL, escapechar='\\')
            saved_files.append(out_path)
            print(f"💾 Saved checkpoint: {out_path}")

            files.download(out_path)

print("\n📄 Files saved:")
for p in saved_files:
    print("  -", p)

✅ Loaded 2166 rows

▶ Strategy=ReACT  |  Backend=Gemini Flash


Gemini Flash/ReACT:   0%|          | 0/566 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()



>>>>>>>> TERMINATING RUN (fd40156c-7d67-47bf-8490-4b9dcaf87d88): Maximum turns (10) reached


Gemini Flash/ReACT:   0%|          | 1/566 [00:28<4:24:53, 28.13s/it]


>>>>>>>> TERMINATING RUN (16c4a81a-d7ec-4b50-b211-0278ae88a25b): Maximum turns (10) reached


...

Gemini Flash/ReACT:  75%|███████▍  | 423/566 [3:10:05<1:00:16, 25.29s/it]


>>>>>>>> TERMINATING RUN (62c1b20b-edae-47cd-972e-5ec2acc38358): Maximum turns (10) reached


Gemini Flash/ReACT:  75%|███████▍  | 424/566 [3:10:28<58:46, 24.84s/it]  


>>>>>>>> TERMINATING RUN (eb279a33-eb69-4f4e-8c46-28e2e3aef1a3): Maximum turns (10) reached


Gemini Flash/ReACT:  75%|███████▌  | 425/566 [3:10:53<58:08, 24.74s/it]